<a href="https://colab.research.google.com/github/divyam-gawde/lightnovel-crawler/blob/dev/NovelFire_WebNovel_EPUB_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title 1 Test Enter Link
link = "https://novelfire.net/book/players-please-board-the-train"  #@param {type:"string", placeholder:"Insert text here"}

import requests
import base64
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# 1) Get the main page
main_page = requests.get(link, headers={"User-Agent": "Mozilla/5.0"})
soup = BeautifulSoup(main_page.text, "lxml")

#--------------------------------------------------------------------------#
#Novel Info
# --- Title ---
title_tag = soup.find("h1", class_="novel-title")
novel_title = title_tag.get_text(strip=True) if title_tag else "Unknown Title"

# --- Authors ---
author_block = soup.find("div", class_="author")
author_spans = author_block.find_all("span", attrs={"itemprop": "author"}) if author_block else []
authors = [span.get_text(strip=True) for span in author_spans]
novel_author = ", ".join(authors) if authors else "Unknown Author"

# --- Identifier from URL ---
identifier = link

# --- Cover image (try to find the main cover <img>) ---
import base64
cover_tag = soup.select_one("figure.cover img")

cover_data = None

if cover_tag and cover_tag.get("src"):
    src = cover_tag["src"]

    if src.startswith("data:"):
        header, b64data = src.split(",", 1)
        cover_data = base64.b64decode(b64data)
    else:
        cover_url = src
        resp = requests.get(cover_url)
        resp.raise_for_status()
        cover_data = resp.content


#--------------------------------------------------------------------------#

# 2) Find the chapter list URL
chapter_link_tag = soup.find("a", class_="grdbtn chapter-latest-container")
chapter_list_url = chapter_link_tag.get("href")

# 3) Load the first chapter list page
chapter_list_page = requests.get(chapter_list_url, headers={"User-Agent": "Mozilla/5.0"})
chapters_soup = BeautifulSoup(chapter_list_page.text, "lxml")

chapters = []

# -------- NEW: detect total pages --------
max_page = 1
for a in chapters_soup.select(".pagination a.page-link"):
    txt = a.get_text(strip=True)
    if txt.isdigit():
        max_page = max(max_page, int(txt))

# -------- NEW: loop through every page --------
for page in range(1, max_page + 1):

    if page == 1:
        page_soup = chapters_soup
    else:
        page_url = f"{chapter_list_url}?page={page}"
        page_req = requests.get(page_url, headers={"User-Agent": "Mozilla/5.0"})
        page_soup = BeautifulSoup(page_req.text, "lxml")

    # Extract chapters from this page
    for a in page_soup.select('#chpagedlist ul.chapter-list li a'):
        url = urljoin(chapter_list_url, a.get("href"))

        no_tag = a.select_one(".chapter-no")
        title_tag = a.select_one(".chapter-title")

        chapter_no = no_tag.get_text(strip=True) if no_tag else ""
        chapter_title = title_tag.get_text(strip=True) if title_tag else ""

        chapters.append({
            "no": chapter_no,
            "title": chapter_title,
            "url": url,
        })

# 5) Downloading Chapters Main Body
for chapter in chapters:
  page = requests.get(chapter["url"], headers={"User-Agent": "Mozilla/5.0"})
  soup = BeautifulSoup(page.text, "lxml")

  container = soup.find("div", id="chapter-container")
  content_div = container.find("div", id="content")

  if content_div:
      chapter["content"] = str(content_div)   # keep HTML for EPUB
  else:
      chapter["content"] = "<p>No content found</p>"

print("Done")

In [ ]:
#@title 1 Test Generate Epub
!pip install -q ebooklib
from ebooklib import epub
import requests

# 1. Create EPUB book object
book = epub.EpubBook()

book.set_identifier(identifier)
book.set_title(novel_title)
book.set_language("en")
book.add_author(novel_author)

# 2. Add cover if we have it
if cover_data:
    book.set_cover("cover.jpg", cover_data)
    print("Cover added to EPUB.")
else:
    print("No cover added (no cover_data).")

# 3. Create chapters and add to the book
epub_chapters = []

for chapter in chapters:
    chapter_html = chapter["content"]
    file_name = f"chapter_{chapter['no']}.xhtml"

    c = epub.EpubHtml(
        title=chapter["title"],
        file_name=file_name,
        lang="en"
    )

    c.content = f"""
    <html xmlns="http://www.w3.org/1999/xhtml">
        <head>
            <title>{chapter['title']}</title>
        </head>
        <body>
            <h2>{chapter['title']}</h2>
            {chapter_html}
        </body>
    </html>
    """

    book.add_item(c)
    epub_chapters.append(c)

# 4. TOC & spine
book.toc = tuple(epub_chapters)
book.spine = ['nav'] + epub_chapters

# 5. Navigation files
book.add_item(epub.EpubNcx())
book.add_item(epub.EpubNav())

# 6. Write EPUB file
safe_title = novel_title.replace(" ", "_")
epub_file_name = f"{safe_title}.epub"
epub.write_epub(epub_file_name, book, {})

print("✔ EPUB successfully created:", epub_file_name)

# -------------------------------------------------------
# 7. Upload EPUB to GoFile
# -------------------------------------------------------

upload_url = "https://upload.gofile.io/uploadfile"

with open(epub_file_name, "rb") as f:
    files = {"file": f}
    response = requests.post(upload_url, files=files)

data = response.json()

if data.get("status") == "ok":
    download_link = data["data"].get("downloadPage")
    direct_link = data["data"].get("directLink")

    print("📤 Uploaded to GoFile!")

    if download_link:
        print("Download page:", download_link)

    if direct_link:
        print("Direct file:", direct_link)

else:
    print("Upload failed:", data)

Cover added to EPUB.
✔ EPUB successfully created: Monarch_of_Solitude:_Daily_Quest_System.epub
📤 Uploaded to GoFile!
Download page: https://gofile.io/d/QG3pfl
